# Regular Positive and Negative Inference (RPNI)

This notebook demonstrates the **RPNI algorithm**, a state-merging algorithm for learning finite-state automata from positive and negative examples.

## Algorithm Overview

RPNI (Regular Positive and Negative Inference) learns a deterministic finite automaton (DFA) by:

1. **Building a prefix tree** from positive examples (words to accept)
2. **Iteratively merging states** that are compatible with both positive and negative examples
3. **Producing a minimal DFA** that generalizes from the training data

The algorithm ensures that:
- All positive examples are accepted
- All negative examples are rejected
- The resulting automaton is as general (minimal) as possible

In [ ]:
from itertools import count

from state_merging.algorithms.rpni import rpni
from state_merging.automata.DFA import DFA, assert_DFA
from state_merging.automata.SFST import run

Create positive examples (strings that should be accepted) and negative examples (strings that should be rejected). The RPNI algorithm uses both to learn a DFA that generalizes correctly.

In [ ]:
input_set: set[int] = {0, 1}

pos_dataset: list[list[int]] = [
    [1],
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1],
    [1, 1, 1],
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1],
    [1, 0, 1, 0, 1]
]

neg_dataset: list[list[int]] = [
    [],
    [0],
    [0, 0],
    [1, 0],
    [0, 1],
    [1, 1],
    [0, 0, 0],
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1]
]

Run the RPNI algorithm with our training data. Then validate that the learned DFA accepts all positive examples and rejects all negative examples.

In [ ]:
dfa: DFA[int, int] = rpni(
    input_set=input_set,
    pos_dataset=pos_dataset,
    neg_dataset=neg_dataset,
    choose_transition=lambda _, trs: next(iter(trs)),
    search_iter=lambda _, qs: qs,
    state_supply=count(),
    verbose=True
)

assert_DFA(dfa)

for d in pos_dataset:
    assert run(dfa, d, None, lambda none, _: none) is not None, \
        f"learned DFA rejected positive data {d}"

for d in neg_dataset:
    assert run(dfa, d, None, lambda none, _: none) is None, \
        f"learned DFA accepted negative data {d}"